In [2]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



In [10]:
sys.path.append("/root/work/tvm-ansor/gallery/constrained_gen")
from modules.task_paths import load_and_register_tasks
tasks = load_and_register_tasks("/root/work/tvm-ansor/gallery/dataset/network_info_all")
# concrete_states = {}
# sketches_by_idx = {}
# for idx, task in enumerate(tasks):
#     concrete_state = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
#     for i, state in enumerate(concrete_state):
#         sketches_by_idx[idx] = (task.desc, f"{task.workload_key}_{i}")
#         concrete_states[f"{task.workload_key}_{i}"] = (task, state)

## 모든 Task

In [18]:
# task.desc를 기준으로 sort
# tasks_sorted = sorted(tasks, key=lambda t: t.desc)
import re

selected_idx = []

for idx, task in enumerate(tasks):
    if (
    task.desc == "vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform"
        or re.fullmatch(
            r"vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_\d+",
            task.desc
        )
    ):
        print(f"Task {idx}: {task.desc}, {task.workload_key}")
        selected_idx.append(idx)
print(selected_idx)

Task 302: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_3, ["72858abe65e3185202b62d45a3956c75", [1, 8, 8, 128], [6, 6, 32, 128], [1, 8, 8, 32]]
Task 313: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_3, ["cdea95b18d7046864bf7e76a2f6b3109", [1, 7, 7, 128], [4, 4, 32, 128], [1, 7, 7, 32]]
Task 375: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_3, ["72858abe65e3185202b62d45a3956c75", [2, 8, 8, 128], [6, 6, 32, 128], [2, 8, 8, 32]]
Task 392: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_3, ["cdea95b18d7046864bf7e76a2f6b3109", [2, 7, 7, 128], [4, 4, 32, 128], [2, 7, 7, 32]]
Task 479: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_2, ["57157118e1492a9a1dd0e0976e6a31b3", [1, 14, 14, 128], [4, 4, 32, 128], [1, 14, 14, 32]]
Task 520: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_3, ["72858abe65e3185202b62d45a3956c75", [4, 8, 8, 128], [6, 6, 32, 128], [4, 8, 8, 32]]
Task 521: vm_mod_f

## Task 묶음

In [30]:
# task를 task.desc 기준으로 sort
tasks = sorted(tasks, key=lambda t: t.desc)
task_set = set(task.desc.rstrip('_0123456789') for task in tasks)
task_set

{'vm_mod_fused_mean',
 'vm_mod_fused_nn_adaptive_avg_pool2d',
 'vm_mod_fused_nn_adaptive_avg_pool3d',
 'vm_mod_fused_nn_avg_pool2d',
 'vm_mod_fused_nn_batch_matmul',
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform',
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu',
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu',
 'vm_mod_fused_nn_conv2d',
 'vm_mod_fused_nn_conv2d_add',
 'vm_mod_fused_nn_conv2d_add_add',
 'vm_mod_fused_nn_conv2d_add_add_clip_divide_multiply',
 'vm_mod_fused_nn_conv2d_add_add_nn_relu',
 'vm_mod_fused_nn_conv2d_add_clip',
 'vm_mod_fused_nn_conv2d_add_clip_divide',
 'vm_mod_fused_nn_conv2d_add_nn_relu',
 'vm_mod_fused_nn_conv2d_transpose',
 'vm_mod_fused_nn_conv3d_add',
 'vm_mod_fused_nn_conv3d_add_add_nn_relu',
 'vm_mod_fused_nn_conv3d_add_nn_relu',
 'vm_mod_fused_nn_dense_add',
 'vm_mod_fused_nn_dense_add_add_clip_divide_multiply',
 'vm_mod_fused_nn_dense_add_fast_tanh',
 'vm_mod_fused

## Task별 DAG

In [35]:
for t in task_set:
    for task in tasks:
        if task.desc.rstrip('_0123456789') == t:
            print(f"{task.desc}, Compute DAG: {task.compute_dag}")
            print("=" * 80)
            break

vm_mod_fused_nn_conv2d_add_add_clip_divide_multiply, Compute DAG: p0 = PLACEHOLDER [1, 224, 224, 3]
pad_temp(i0, i1, i2, i3) = tir.if_then_else(((((i1 >= 1) && (i1 < 225)) && (i2 >= 1)) && (i2 < 225)), p0[i0, (i1 - 1), (i2 - 1), i3], 0f)
p1 = PLACEHOLDER [3, 3, 3, 16]
conv2d_nhwc(nn, yy, xx, ff) += (pad_temp[nn, ((yy*2) + ry), ((xx*2) + rx), rc]*p1[ry, rx, rc, ff])
p2 = PLACEHOLDER [1, 1, 1, 16]
T_add(ax0, ax1, ax2, ax3) = (conv2d_nhwc[ax0, ax1, ax2, ax3] + p2[ax0, 0, 0, ax3])
compile_engine_const() = 3f
T_add(ax0, ax1, ax2, ax3) = (T_add[ax0, ax1, ax2, ax3] + compile_engine_const[])
compute(i0, i1, i2, i3) = max(min(T_add[i0, i1, i2, i3], 6f), 0f)
compile_engine_const() = 6f
T_divide(ax0, ax1, ax2, ax3) = (compute[ax0, ax1, ax2, ax3]/compile_engine_const[])
T_multiply(ax0, ax1, ax2, ax3) = (T_add[ax0, ax1, ax2, ax3]*T_divide[ax0, ax1, ax2, ax3])

vm_mod_fused_nn_conv2d, Compute DAG: p0 = PLACEHOLDER [1, 56, 56, 256]
pad_temp(i0, i1, i2, i3) = p0[i0, i1, i2, i3]
p1 = PLACEHOLDER [1, 1,

## Task별 sketch 개수

In [6]:
task_and_count = {}
index_and_sketches = {}

for idx, task in enumerate(tasks):
    sketches = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
    sketch_count = len(sketches)
    if task.desc not in task_and_count:
        task_and_count[task.desc] = []
        index_and_sketches[task.desc] = []
        task_and_count[task.desc].append(sketch_count)
        index_and_sketches[task.desc].append((idx, sketch_count, sketches))

    elif task.desc in task_and_count:
        if sketch_count not in task_and_count[task.desc]:
            task_and_count[task.desc].append(sketch_count)
            index_and_sketches[task.desc].append((idx, sketch_count, sketches))
task_and_count

{'vm_mod_fused_mean': [2],
 'vm_mod_fused_nn_adaptive_avg_pool2d': [1],
 'vm_mod_fused_nn_adaptive_avg_pool2d_1': [1],
 'vm_mod_fused_nn_adaptive_avg_pool2d_2': [1],
 'vm_mod_fused_nn_adaptive_avg_pool2d_3': [1],
 'vm_mod_fused_nn_adaptive_avg_pool2d_4': [1],
 'vm_mod_fused_nn_adaptive_avg_pool2d_5': [1],
 'vm_mod_fused_nn_adaptive_avg_pool3d': [1],
 'vm_mod_fused_nn_avg_pool2d': [1],
 'vm_mod_fused_nn_avg_pool2d_1': [1],
 'vm_mod_fused_nn_avg_pool2d_2': [1],
 'vm_mod_fused_nn_avg_pool2d_3': [1],
 'vm_mod_fused_nn_avg_pool2d_4': [1],
 'vm_mod_fused_nn_avg_pool2d_5': [1],
 'vm_mod_fused_nn_batch_matmul': [1],
 'vm_mod_fused_nn_batch_matmul_1': [1],
 'vm_mod_fused_nn_batch_matmul_2': [1],
 'vm_mod_fused_nn_batch_matmul_3': [1],
 'vm_mod_fused_nn_batch_matmul_4': [1],
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform': [1],
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_1': [1],
 'vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_2': [1]

In [7]:
# 같은 task.desc 안에서 sketch_count가 서로 다른 경우만 확인
sketch_count_diffs = {
    desc: sorted(entries, key=lambda x: (x[1], x[0]))
    for desc, entries in index_and_sketches.items()
    if len({sketch_count for _, sketch_count, _ in entries}) > 1
}

sketch_count_diff_summary = []
for desc, entries in sketch_count_diffs.items():
    sketch_count_diff_summary.append({
        "task_desc": desc,
        "sketch_counts": [sketch_count for _, sketch_count, _ in entries],
        "task_indices": [idx for idx, _, _ in entries],
        "workload_keys": [tasks[idx].workload_key for idx, _, _ in entries],
    })


def _get_sketches_for_task_idx(task_idx):
    task = tasks[task_idx]
    for idx, sketch_count, sketches in index_and_sketches.get(task.desc, []):
        if idx == task_idx:
            return sketch_count, sketches

    sketches = SketchPolicy(
        task,
        params={'sample_init_no_invalid': 1},
        verbose=False,
    ).generate_concrete_sketches()
    return len(sketches), sketches


def show_task_sketches(*task_indices, max_sketches=None):
    """Print all sketches for the given task indices."""
    for task_idx in task_indices:
        task = tasks[task_idx]
        sketch_count, sketches = _get_sketches_for_task_idx(task_idx)
        shown_sketches = sketches if max_sketches is None else sketches[:max_sketches]

        print(f"Task {task_idx}: {task.desc}")
        print(f"workload_key: {task.workload_key}")
        print(f"sketch_count: {sketch_count}")
        print("-" * 80)
        for sketch_idx, sketch in enumerate(shown_sketches):
            print(f"sketch[{sketch_idx}]")
            print(sketch)
            print("-" * 80)
        if max_sketches is not None and len(sketches) > max_sketches:
            print(f"... {len(sketches) - max_sketches} more sketches")
        print("=" * 80)


def show_sketch_count_diff(task_desc=None, max_sketches=None):
    """Print every mismatched task entry and its sketches."""
    items = sketch_count_diffs.items()
    if task_desc is not None:
        items = [(task_desc, sketch_count_diffs[task_desc])]

    for desc, entries in items:
        print(f"task.desc: {desc}")
        for task_idx, _, _ in entries:
            show_task_sketches(task_idx, max_sketches=max_sketches)

# 예시: 1225와 1224의 sketch 전체 출력
show_task_sketches(1225, 1224)

# mismatch 목록만 보고 싶으면 아래를 실행
# sketch_count_diff_summary


Task 1225: vm_mod_fused_nn_conv2d_add_nn_relu_10
workload_key: ["e0913b23dc38a8096f5015fc6aaa1ed3", [4, 1, 1, 480], [1, 1, 480, 120], [1, 1, 1, 120], [4, 1, 1, 120]]
sketch_count: 1
--------------------------------------------------------------------------------
sketch[0]
Placeholder: p0, p1, p2
blockIdx.x ax0.0@ax1.0@ax2.0@ax3.0@ (0,10)
  vthread ax0.1@ax1.1@ax2.1@ax3.1@ (0,4)
    threadIdx.x ax0.2@ax1.2@ax2.2@ax3.2@ (0,4)
      conv2d_nhwc auto_unroll: 512
      for nn.0 (0,1)
        for yy.0 (0,1)
          for xx.0 (0,1)
            for ff.0 (0,1)
              for nn.1 (0,1)
                for yy.1 (0,1)
                  for xx.1 (0,1)
                    for ff.1 (0,1)
                      for nn.2 (0,1)
                        for yy.2 (0,1)
                          for xx.2 (0,1)
                            for ff.2 (0,1)
                              for ry.0 (0,1)
                                for rx.0 (0,1)
                                  for rc.0 (0,12)
           

In [ ]:
for idx, task in enumerate(tasks):
    print(f"Task {idx}: {task.desc}, {task.workload_key}")
    print(task.compute_dag)

Task 0: vm_mod_fused_mean, ["e080f236e50610bed197d39fac1f5797", [1, 64, 512], [1, 64, 1]]
p0 = PLACEHOLDER [1, 64, 512]
p0_red(ax0, ax1, ax2) += p0[ax0, ax1, k2]
T_divide(ax0, ax1, ax2) = (p0_red[ax0, ax1, ax2]/512f)

Task 1: vm_mod_fused_mean, ["94ea9b2f662177038f4761f5e1fdc14b", [1, 64, 768], [1, 64, 1]]
p0 = PLACEHOLDER [1, 64, 768]
p0_red(ax0, ax1, ax2) += p0[ax0, ax1, k2]
T_divide(ax0, ax1, ax2) = (p0_red[ax0, ax1, ax2]/768f)

Task 2: vm_mod_fused_mean, ["80312e579af64168aa60737451fa5268", [1, 64, 1024], [1, 64, 1]]
p0 = PLACEHOLDER [1, 64, 1024]
p0_red(ax0, ax1, ax2) += p0[ax0, ax1, k2]
T_divide(ax0, ax1, ax2) = (p0_red[ax0, ax1, ax2]/1024f)

Task 3: vm_mod_fused_mean, ["e080f236e50610bed197d39fac1f5797", [1, 128, 512], [1, 128, 1]]
p0 = PLACEHOLDER [1, 128, 512]
p0_red(ax0, ax1, ax2) += p0[ax0, ax1, k2]
T_divide(ax0, ax1, ax2) = (p0_red[ax0, ax1, ax2]/512f)

Task 4: vm_mod_fused_mean, ["e080f236e50610bed197d39fac1f5797", [2, 64, 512], [2, 64, 1]]
p0 = PLACEHOLDER [2, 64, 512]
p0

In [9]:
ENABLED_CONSTRAINTS_NO_VECTORIZE = (
    'shared_memory',
    'vectorize',
    'max_threads',
    'max_vthread',
    'innermost_split',
    'split_structure',
)

sys.path.append("/root/work/tvm-ansor/gallery/constrained_gen")
from modules.symbolic_state_bridge import build_symbolic_state
from modules.schedule_generator import ScheduleGenerator
task_idx = 1224
task = tasks[task_idx]
sample = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()[1]
sym_state = build_symbolic_state(task, sample)
sg = ScheduleGenerator(sym_state, task=task, base_state=sample, enabled_constraints=ENABLED_CONSTRAINTS_NO_VECTORIZE)
print(task_idx, ":", task.desc)
print(task.workload_key)
print(sym_state)
# cs.transform_steps[-5].extent


1224 : vm_mod_fused_nn_conv2d_add_nn_relu_10
["f5b447dcba2b43cc270d7c2062043946", [1, 1, 1, 480], [1, 1, 480, 120], [1, 1, 1, 120], [1, 1, 1, 120]]
Placeholder: p0, p1, p2
blockIdx.x ax0.0@ax1.0@ax2.0@ax3.0@ (0,ceil(ceil(ceil(1/(min(sp_1_2*sp_1_3,1)))/(min(sp_1_1,ceil(1/(min(sp_1_2*sp_1_3,1))))))/(min(sp_1_0,ceil(ceil(1/(min(sp_1_2*sp_1_3,1)))/(min(sp_1_1,ceil(1/(min(sp_1_2*sp_1_3,1)))))))))*ceil(ceil(ceil(1/(min(sp_2_2*sp_2_3,1)))/(min(sp_2_1,ceil(1/(min(sp_2_2*sp_2_3,1))))))/(min(sp_2_0,ceil(ceil(1/(min(sp_2_2*sp_2_3,1)))/(min(sp_2_1,ceil(1/(min(sp_2_2*sp_2_3,1)))))))))*ceil(ceil(ceil(1/(min(sp_3_2*sp_3_3,1)))/(min(sp_3_1,ceil(1/(min(sp_3_2*sp_3_3,1))))))/(min(sp_3_0,ceil(ceil(1/(min(sp_3_2*sp_3_3,1)))/(min(sp_3_1,ceil(1/(min(sp_3_2*sp_3_3,1)))))))))*ceil(ceil(ceil(120/(min(sp_4_2*sp_4_3,120)))/(min(sp_4_1,ceil(120/(min(sp_4_2*sp_4_3,120))))))/(min(sp_4_0,ceil(ceil(120/(min(sp_4_2*sp_4_3,120)))/(min(sp_4_1,ceil(120/(min(sp_4_2*sp_4_3,120))))))))))
  vthread ax0.1@ax1.1@ax2.1@ax3.1@ (

In [21]:
task_idx = 2268
task = tasks[task_idx]
sample = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()[0]
sym_state = build_symbolic_state(task, sample)
sg = ScheduleGenerator(sym_state, task=task, base_state=sample, enabled_constraints=ENABLED_CONSTRAINTS_NO_VECTORIZE)
print(task_idx, ":", task.desc)
print(task.workload_key)
print(sym_state)

2268 : vm_mod_fused_nn_conv3d_add_nn_relu_5
["08978146362a6a5b88823a549d373000", [1, 28, 14, 2, 256], [3, 3, 3, 256, 256], [1, 1, 1, 1, 256], [1, 28, 14, 2, 256]]
Placeholder: p0, p1, p2
blockIdx.x ax0.0@ax1.0@ax2.0@ax3.0@ax4.0@ (0,ceil(ceil(ceil(1/(min(sp_1_2*sp_1_3,1)))/(min(sp_1_1,ceil(1/(min(sp_1_2*sp_1_3,1))))))/(min(sp_1_0,ceil(ceil(1/(min(sp_1_2*sp_1_3,1)))/(min(sp_1_1,ceil(1/(min(sp_1_2*sp_1_3,1)))))))))*ceil(ceil(ceil(28/(min(sp_2_2*sp_2_3,28)))/(min(sp_2_1,ceil(28/(min(sp_2_2*sp_2_3,28))))))/(min(sp_2_0,ceil(ceil(28/(min(sp_2_2*sp_2_3,28)))/(min(sp_2_1,ceil(28/(min(sp_2_2*sp_2_3,28)))))))))*ceil(ceil(ceil(14/(min(sp_3_2*sp_3_3,14)))/(min(sp_3_1,ceil(14/(min(sp_3_2*sp_3_3,14))))))/(min(sp_3_0,ceil(ceil(14/(min(sp_3_2*sp_3_3,14)))/(min(sp_3_1,ceil(14/(min(sp_3_2*sp_3_3,14)))))))))*ceil(ceil(ceil(2/(min(sp_4_2*sp_4_3,2)))/(min(sp_4_1,ceil(2/(min(sp_4_2*sp_4_3,2))))))/(min(sp_4_0,ceil(ceil(2/(min(sp_4_2*sp_4_3,2)))/(min(sp_4_1,ceil(2/(min(sp_4_2*sp_4_3,2)))))))))*ceil(ceil(ceil(2

In [ ]:
sg.s._exception_split_names

In [6]:
print(sg._get_constraints_str())

[grid_1: blockIdx.x@s4.i0]
  max_threads: min(sp_60_0,ceil(196/(min(sp_27_0,196)))*ceil(128/(min(sp_28_0,128)))*min(sp_27_0,196)*min(sp_28_0,128)) <= 1024

[grid_2: blockIdx.x@s9.i0]
  max_threads:
    min(sp_9_1,ceil(6/(min(sp_9_2*sp_9_3,6))))
    * min(sp_10_1,ceil(6/(min(sp_10_2*sp_10_3,6))))
    * min(sp_11_1,ceil(196/(min(sp_11_2*sp_11_3,196))))
    * min(sp_12_1,ceil(32/(min(sp_12_2*sp_12_3,32))))
    <= 1024
  max_vthread:
    min(sp_9_0,ceil(ceil(6/(min(sp_9_2*sp_9_3,6)))/(min(sp_9_1,ceil(6/(min(sp_9_2*sp_9_3,6)))))))
    * min(sp_10_0,ceil(ceil(6/(min(sp_10_2*sp_10_3,6)))/(min(sp_10_1,ceil(6/(min(sp_10_2*sp_10_3,6)))))))
    * min(sp_11_0,ceil(ceil(196/(min(sp_11_2*sp_11_3,196)))/(min(sp_11_1,ceil(196/(min(sp_11_2*sp_11_3,196)))))))
    * min(sp_12_0,ceil(ceil(32/(min(sp_12_2*sp_12_3,32)))/(min(sp_12_1,ceil(32/(min(sp_12_2*sp_12_3,32)))))))
    <= 8

[grid_3: blockIdx.x@s11.i0]
  max_threads: min(sp_40_0,ceil(196/(min(sp_2_0,196)))*ceil(32/(min(sp_3_0,32)))*min(sp_2_0,196)*min

In [7]:
sg._get_var_order_phase_entries()

[{'phase_name': 'grid_0__memory_split_structure',
  'phase_family': 'memory_split_structure',
  'grid_scope': (),
  'param_names': ['sp_9_3',
   'sp_9_2',
   'sp_9_1',
   'sp_9_0',
   'sp_10_3',
   'sp_10_2',
   'sp_10_1',
   'sp_10_0',
   'sp_11_3',
   'sp_11_2',
   'sp_11_1',
   'sp_11_0',
   'sp_12_3',
   'sp_12_2',
   'sp_12_1',
   'sp_12_0',
   'sp_13_1',
   'sp_13_0',
   'sp_27_0',
   'sp_28_0',
   'sp_60_0'],
  'param_entries': [{'param_name': 'sp_9_3',
    'param_kind': 'split',
    'split_step_idx': 9,
    'split_position': 3,
    'split_extent': 6,
    'split_group_param_names': ['sp_9_0', 'sp_9_1', 'sp_9_2', 'sp_9_3'],
    'collapsed_factor_param_names': ['sp_9_0', 'sp_9_1', 'sp_9_2'],
    'is_innermost': True},
   {'param_name': 'sp_9_2',
    'param_kind': 'split',
    'split_step_idx': 9,
    'split_position': 2,
    'split_extent': 6,
    'split_group_param_names': ['sp_9_0', 'sp_9_1', 'sp_9_2', 'sp_9_3'],
    'collapsed_factor_param_names': ['sp_9_0', 'sp_9_1'],
    'is_

In [8]:
sg.get_full_var_order_entries()

{'phase_count': 3,
 'param_order': ['sp_9_3',
  'sp_9_2',
  'sp_9_1',
  'sp_9_0',
  'sp_10_3',
  'sp_10_2',
  'sp_10_1',
  'sp_10_0',
  'sp_11_3',
  'sp_11_2',
  'sp_11_1',
  'sp_11_0',
  'sp_12_3',
  'sp_12_2',
  'sp_12_1',
  'sp_12_0',
  'sp_13_1',
  'sp_13_0',
  'sp_27_0',
  'sp_28_0',
  'sp_60_0',
  'sp_50_0',
  'sp_55_0',
  'ur_63',
  'ur_64',
  'ur_65',
  'sp_2_0',
  'sp_3_0',
  'sp_36_0',
  'sp_40_0'],
 'phases': [{'phase_name': 'grid_0__memory_split_structure',
   'phase_family': 'memory_split_structure',
   'grid_scope': (),
   'param_names': ['sp_9_3',
    'sp_9_2',
    'sp_9_1',
    'sp_9_0',
    'sp_10_3',
    'sp_10_2',
    'sp_10_1',
    'sp_10_0',
    'sp_11_3',
    'sp_11_2',
    'sp_11_1',
    'sp_11_0',
    'sp_12_3',
    'sp_12_2',
    'sp_12_1',
    'sp_12_0',
    'sp_13_1',
    'sp_13_0',
    'sp_27_0',
    'sp_28_0',
    'sp_60_0'],
   'param_entries': [{'param_name': 'sp_9_3',
     'param_kind': 'split',
     'split_step_idx': 9,
     'split_position': 3,
     's

In [13]:
sg.get_constraints_under_assignment()

{'assignment': {'params': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1}},
 'domains': {'all': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1,
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
   'sp_6_0': [1, 3],
   'sp_6_1': [1, 3],
   'sp_26_0': [1, 36],
   'sp_31_0': [1, 64]},
  'fixed': {'sp_1_0': 1, 'sp_1_1': 1, 'sp_1_2': 1, 'sp_1_3': 1},
  'remaining': {'sp_26_0': [1, 36],
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_31_0': [1, 64],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
 

In [4]:
sg.get_param_candidates(name='sp_31_0')

{'assignment': {'params': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1}},
 'domains': {'all': {'sp_1_0': 1,
   'sp_1_1': 1,
   'sp_1_2': 1,
   'sp_1_3': 1,
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
   'sp_6_0': [1, 3],
   'sp_6_1': [1, 3],
   'sp_26_0': [1, 36],
   'sp_31_0': [1, 60]},
  'fixed': {'sp_1_0': 1, 'sp_1_1': 1, 'sp_1_2': 1, 'sp_1_3': 1},
  'remaining': {'sp_26_0': [1, 36],
   'sp_2_0': [1, 8],
   'sp_2_1': [1, 112],
   'sp_2_2': [1, 112],
   'sp_2_3': [1, 56],
   'sp_31_0': [1, 60],
   'sp_3_0': [1, 8],
   'sp_3_1': [1, 112],
   'sp_3_2': [1, 112],
   'sp_3_3': [1, 56],
   'sp_4_0': [1, 8],
   'sp_4_1': [1, 32],
   'sp_4_2': [1, 32],
   'sp_4_3': [1, 32],
   'sp_5_0': [1, 3],
   'sp_5_1': [1, 3],
 